In [8]:
import re
import sys
import time
import random

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

UA = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}

BB_CATS = {
    "/catalog/roza/": 4,
    "/catalog/hrizantema/": 3,
    "/catalog/tyulpan/": 3,
    "/catalog/pion/": 2,
    "/catalog/liliya/": 2,
    "/catalog/gerbera/": 2,
    "/catalog/alstromeriya/": 2,
    "/catalog/eustoma/": 2,
}

SKIP_OPTCVET = {"корзина", "цимбидиум", "орхидея", "орхид"}
MAX_PER_PIECE = 600


def fetch(url, pause=True):
    if pause:
        time.sleep(random.uniform(1.0, 2.0))
    s = requests.Session()
    s.headers.update(UA)
    r = s.get(url, timeout=15)
    if r.status_code != 200:
        print(f"Упс, {r.status_code} на {url}")
        return None
    return BeautifulSoup(r.text, "html.parser")


def rub(x, _=None):
    return f"{int(x):,}".replace(",", "\u202f")

#megacvet

MEGACVET_SECTIONS = [
    "https://megacvet24.ru/rozy/",
    "https://megacvet24.ru/krasnye-rozy/",
    "https://megacvet24.ru/belye-rozy/",
    "https://megacvet24.ru/zheltye-rozy/",
    "https://megacvet24.ru/piony/",
    "https://megacvet24.ru/gortenzii/",
    "https://megacvet24.ru/lilii/",
    "https://megacvet24.ru/hrizantemy/",
    "https://megacvet24.ru/alstromerii/",
    "https://megacvet24.ru/irisy/",
    "https://megacvet24.ru/gerbery/",
    "https://megacvet24.ru/eustomy/",
    "https://megacvet24.ru/gvozdiki/",
    "https://megacvet24.ru/podsolnuhi/",
    "https://megacvet24.ru/romashki/",
    "https://megacvet24.ru/orhidei/",
    "https://megacvet24.ru/orhideya-dendrobium/",
    "https://megacvet24.ru/tulpany/",
    "https://megacvet24.ru/gipsofila/",
    "https://megacvet24.ru/verba/",
]


def is_mono(name):
    if not re.match(r"^\d+\s+", name):
        return False
    if " и " in name.lower():
        return False
    return True


def price_per_15(name, price):
    m = re.match(r"^(\d+)\s+", name)
    if not m:
        return None
    count = int(m.group(1))
    if count == 0:
        return None
    return round(price / count * 15)


def cheapest_mono_from_section(url):
    candidates = []

    for page in range(1, 6):
        paged = url if page == 1 else f"{url}?page={page}"
        soup = fetch(paged)
        if not soup:
            break

        cards = soup.select("div.list-product")
        if not cards:
            break

        for card in cards:
            name_el = card.select_one("a.list-product__name span") \
                   or card.select_one("a.list-product__name")
            name = name_el.get_text(strip=True) if name_el else ""
            if not is_mono(name):
                continue

            price = None
            meta = card.select_one('meta[itemprop="price"]')
            if meta and meta.get("content"):
                try:
                    price = int(float(meta["content"]))
                except ValueError:
                    pass

            if not price:
                span = card.select_one("span.list-product__price")
                if span:
                    for junk in span.select("span"):
                        junk.decompose()
                    digits = re.sub(r"[^\d]", "", span.get_text())
                    if digits:
                        price = int(digits)

            if not price or price < 200 or price > 500_000:
                continue

            p15 = price_per_15(name, price)
            if not p15:
                continue

            m = re.match(r"^(\d+)\s+", name)
            count = m.group(1) if m else "?"
            label = name[:50] + (" [×15]" if count != "15" else "")
            candidates.append({"name": label[:60], "price": p15})

        if not soup.select_one("a[rel='next'], a[class*='next'], .pagination__next"):
            break

    if not candidates:
        return None
    return min(candidates, key=lambda r: r["price"])


def scrape_megacvet():
    result = []
    seen_names = set()

    for url in MEGACVET_SECTIONS:
        row = cheapest_mono_from_section(url)
        if row and row["name"] not in seen_names:
            seen_names.add(row["name"])
            result.append(row)

    return result

#bbflowers

def bbflowers_price(item_url):
    soup = fetch(item_url)
    if not soup:
        return None
    text = soup.get_text(" ", strip=True)
    m = re.search(r"(\d+[.,]\d+)\s*р\.\s*за\s*1\s*шт", text)
    if not m:
        return None
    try:
        val = float(m.group(1).replace(",", "."))
        return val if 10 <= val <= 50_000 else None
    except ValueError:
        return None


def bbflowers_category_links(cat, limit):
    soup = fetch("https://bbflowers.ru" + cat)
    if not soup:
        return []

    links = []
    for card in soup.select("div.item"):
        a = card.select_one("div.title a") or card.select_one("a")
        if not a:
            continue
        href = a.get("href", "")
        name = a.get_text(strip=True)
        if not href or not name:
            continue
        if not href.startswith("http"):
            href = "https://bbflowers.ru" + href
        links.append((name[:60], href))

    if not links:
        return []

    n = len(links)
    limit = min(limit, n)
    step = n / limit
    return [links[int(round(i * step))] for i in range(limit)]


def scrape_bbflowers():
    links = []
    for cat, limit in BB_CATS.items():
        links.extend(bbflowers_category_links(cat, limit))

    result = []
    for name, href in links:
        per_piece = bbflowers_price(href)
        if per_piece:
            result.append({"name": name, "price": round(per_piece * 15)})

    return list({(r["name"], r["price"]): r for r in result}.values())

#optcvet

OPTCVET_TYPES = [
    ("роза",        "Роза"),
    ("тюльпан",     "Тюльпан"),
    ("гербер",      "Гербера"),
    ("альстромери", "Альстромерия"),
    ("хризантем",   "Хризантема"),
    ("дендробиум",  "Дендробиум"),
    ("гвоздик",     "Гвоздика"),
    ("ирис",        "Ирис"),
]


def scrape_optcvet():
    soup = fetch("https://cvetochnay-baza.ru/catalog.html")
    if not soup:
        return []

    all_items = []
    for card in soup.select("div.product_list_item"):
        name_el = card.select_one("div.product_list_name a") or card.select_one("a")
        name = name_el.get_text(strip=True) if name_el else ""
        if not name:
            continue
        if any(skip in name.lower() for skip in SKIP_OPTCVET):
            continue

        card_text = card.get_text(" ", strip=True)
        m = re.search(r"[Цц]ена\s*:?\s*(?:от\s*)?(\d[\d\s]*)\s*руб", card_text)
        if not m:
            continue

        per_piece = int(re.sub(r"\s", "", m.group(1)))
        if per_piece > MAX_PER_PIECE or per_piece < 50:
            continue

        all_items.append({"name": name, "per_piece": per_piece})

    result = []
    for ftype, label in OPTCVET_TYPES:
        candidates = [i for i in all_items if ftype in i["name"].lower()]
        if not candidates:
            continue
        cheapest = min(candidates, key=lambda i: i["per_piece"])
        result.append({"name": cheapest["name"][:60], "price": cheapest["per_piece"] * 15})

    return result

#figures

def shop_chart(rows, title, color, fname):
    df = pd.DataFrame(rows).sort_values("price").reset_index(drop=True)
    n = len(df)
    mean_v = df["price"].mean()
    med_v  = df["price"].median()
    min_v  = df["price"].min()
    max_v  = df["price"].max()

    fig, (left, right) = plt.subplots(
        1, 2, figsize=(18, max(7, n * 0.42)),
        gridspec_kw={"width_ratios": [3, 1]},
    )
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)

    bars = left.barh(df["name"], df["price"], color=color, edgecolor="white", height=0.72)
    left.set_xlabel("Цена, руб.", fontsize=11)
    left.set_title("Название и цена", fontsize=12, fontweight="bold")
    left.xaxis.set_major_formatter(mticker.FuncFormatter(rub))
    left.grid(axis="x", alpha=0.25)
    left.set_axisbelow(True)
    left.tick_params(axis="y", labelsize=8.5)

    gap = max_v * 0.008
    for bar, val in zip(bars, df["price"]):
        left.text(val + gap, bar.get_y() + bar.get_height() / 2,
                  f"{val:,}\u202f₽".replace(",", "\u202f"),
                  va="center", fontsize=8)

    left.axvline(mean_v, color="#E76F51", lw=1.6, ls="--",
                 label=f"Среднее\u2002{mean_v:,.0f}\u202f₽".replace(",", "\u202f"))
    left.axvline(med_v,  color="#264653", lw=1.6, ls=":",
                 label=f"Медиана\u2002{med_v:,.0f}\u202f₽".replace(",", "\u202f"))
    left.legend(fontsize=9, loc="lower right")

    stat_bars = right.bar(["Среднее", "Медиана"], [mean_v, med_v],
                          color=["#E76F51", "#264653"], edgecolor="white", width=0.45)
    right.set_ylabel("Цена, руб.", fontsize=11)
    right.set_title("Среднее и медиана", fontsize=12, fontweight="bold")
    right.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
    right.grid(axis="y", alpha=0.25)
    right.set_axisbelow(True)

    top = max_v * 0.025
    for bar, val in zip(stat_bars, [mean_v, med_v]):
        right.text(bar.get_x() + bar.get_width() / 2, val + top,
                   f"{val:,.0f}\u202f₽".replace(",", "\u202f"),
                   ha="center", fontsize=12, fontweight="bold")

    note = (f"Позиций:  {n}\n"
            f"Мин:  {min_v:,}\u202f₽\n".replace(",", "\u202f") +
            f"Макс: {max_v:,}\u202f₽".replace(",", "\u202f"))
    right.text(0.5, 0.06, note, transform=right.transAxes,
               ha="center", va="bottom", fontsize=9,
               bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f9f4", edgecolor=color))

    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()


def comparison_chart(shops):
    shops = [s for s in shops if s["rows"]]
    if not shops:
        return

    labels = [s["label"] for s in shops]
    colors = [s["color"] for s in shops]
    data   = [pd.Series([r["price"] for r in s["rows"]]) for s in shops]

    means   = [d.mean()   for d in data]
    medians = [d.median() for d in data]
    mins    = [d.min()    for d in data]
    maxs    = [d.max()    for d in data]
    stds    = [d.std()    for d in data]
    counts  = [len(d)     for d in data]

    short = [l.split("—")[0].strip().split("(")[0].strip() for l in labels]

    fig = plt.figure(figsize=(18, 14))
    fig.suptitle("Сравнение цен на букеты из 15 цветов — все три магазина",
                 fontsize=15, fontweight="bold", y=1.01)
    gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    bars = ax.bar(short, means, color=colors, edgecolor="white", width=0.5)
    ax.set_title("Средняя цена букета", fontsize=12, fontweight="bold")
    ax.set_ylabel("Руб.")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, val + max(means) * 0.02,
                f"{val:,.0f}\u202f₽".replace(",", "\u202f"),
                ha="center", fontsize=11, fontweight="bold")

    ax = fig.add_subplot(gs[0, 1])
    bars = ax.bar(short, medians, color=colors, edgecolor="white", width=0.5)
    ax.set_title("Медианная цена букета", fontsize=12, fontweight="bold")
    ax.set_ylabel("Руб.")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    for bar, val in zip(bars, medians):
        ax.text(bar.get_x() + bar.get_width() / 2, val + max(medians) * 0.02,
                f"{val:,.0f}\u202f₽".replace(",", "\u202f"),
                ha="center", fontsize=11, fontweight="bold")

    ax = fig.add_subplot(gs[1, 0])
    bp = ax.boxplot([d.tolist() for d in data], tick_labels=short,
                    patch_artist=True, widths=0.45,
                    medianprops=dict(color="white", linewidth=2))
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.82)
    for elem in ["whiskers", "caps", "fliers"]:
        for line in bp[elem]:
            line.set_color("#555")
    ax.set_title("Разброс цен", fontsize=12, fontweight="bold")
    ax.set_ylabel("Руб.")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(rub))
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    ax = fig.add_subplot(gs[1, 1])
    ax.axis("off")
    ax.set_title("Ключевые метрики", fontsize=12, fontweight="bold")

    headers = ["Магазин", "Поз.", "Мин ₽", "Среднее ₽", "Медиана ₽", "Макс ₽", "Разброс ₽"]
    rows = []
    for i in range(len(shops)):
        rows.append([
            short[i],
            str(counts[i]),
            f"{int(mins[i]):,}".replace(",", "\u202f"),
            f"{int(means[i]):,}".replace(",", "\u202f"),
            f"{int(medians[i]):,}".replace(",", "\u202f"),
            f"{int(maxs[i]):,}".replace(",", "\u202f"),
            f"{int(stds[i]):,}".replace(",", "\u202f"),
        ])

    tbl = ax.table(cellText=rows, colLabels=headers,
                   cellLoc="center", loc="upper center",
                   bbox=[0, 0.38, 1, 0.58])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    for j in range(len(headers)):
        tbl[0, j].set_facecolor("#2D6A4F")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        bg = "#f0f9f4" if i % 2 == 0 else "white"
        for j in range(len(headers)):
            tbl[i, j].set_facecolor(bg)

    cheapest = short[means.index(min(means))]
    mid_shop = short[medians.index(min(medians))]
    widest   = short[stds.index(max(stds))]

    notes = [
        f"Самая низкая средняя цена: {cheapest}",
        f"Самая низкая медиана: {mid_shop}",
        f"Наибольший разброс: {widest}",
        f"(широкий разброс = разные ценовые сегменты)",
        "",
        "Вывод по закупке:",
        f"Бюджетный букет — смотреть на {mid_shop}",
        f"Разброс цен показывает вашу конкурентную нишу",
    ]
    y = 0.33
    for line in notes:
        bold  = "bold"    if line.startswith("Вывод") else "normal"
        color = "#1B4332" if line.startswith("Вывод") else "#333"
        ax.text(0.02, y, line, transform=ax.transAxes,
                fontsize=8.5, va="top", color=color, fontweight=bold)
        y -= 0.075

    plt.savefig("04_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()


def main():
    megacvet = scrape_megacvet()
    bbflowers = scrape_bbflowers()
    optcvet = scrape_optcvet()

    if not any([megacvet, bbflowers, optcvet]):
        print("Ни один сайт не ответил — проверь соединение")
        sys.exit(1)

    all_data = (
        [{"source": "МегаЦвет24", **r} for r in megacvet]  +
        [{"source": "BBFlowers ×15", **r} for r in bbflowers] +
        [{"source": "ОптЦвет ×15", **r} for r in optcvet]
    )
    pd.DataFrame(all_data).to_csv("bouquets_data.csv", index=False, encoding="utf-8-sig")

    if megacvet:
        shop_chart(megacvet, "МегаЦвет24 — букеты из 15 цветов",  "#2D6A4F", "01_megacvet.png")
    if bbflowers:
        shop_chart(bbflowers,"BBFlowers — цветы поштучно × 15",    "#E76F51", "02_bbflowers.png")
    if optcvet:
        shop_chart(optcvet, "ОптЦвет — цветы поштучно × 15",     "#74C69D", "03_optcvet.png")

    comparison_chart([
        {"label": "МегаЦвет24", "rows": megacvet,  "color": "#2D6A4F"},
        {"label": "BBFlowers ×15", "rows": bbflowers, "color": "#E76F51"},
        {"label": "ОптЦвет ×15", "rows": optcvet,   "color": "#74C69D"},
    ])

    print("Созданные файлы:")
    print("bouquets_data.csv")
    if megacvet: print("01_megacvet.png")
    if bbflowers: print("02_bbflowers.png")
    if optcvet: print("03_optcvet.png")
    print("04_comparison.png")


if __name__ == "__main__":
    main()

Упс, 404 на https://megacvet24.ru/tulpany/
Созданные файлы:
bouquets_data.csv
01_megacvet.png
02_bbflowers.png
03_optcvet.png
04_comparison.png
